# 🏠 Phase 1: Real Estate Data Scraping Notebook

In this notebook, we use the `homeharvest` library to fetch real-time and sold housing listings from major US real estate sources (Zillow, Redfin, Realtor.com) and store them into our SQLite database (`data/database.sqlite`).

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import sqlite3
from src.scraper.homeharvest_scraper import run_scraper
from src.utils.db_manager import upsert_properties, load_all_properties, count_properties

In [ ]:
# لیست کدهای پستی معروف شهر آستین، تگزاس
austin_zip_codes = [
    "78701", "78702", "78703", "78704", "78705", 
    "78741", "78744", "78745", "78750", "78759",
    "78717", "78723", "78727", "78731", "78735"
]

all_scraped_data = []

print("🚀 شروع فرآیند استخراج داده‌های انبوه...")

for zip_code in austin_zip_codes:
    print(f"\n🔍 در حال اسکرپ کد پستی {zip_code}...")
    try:
        # برای هر کد پستی، ۱۰۰ خانه در حال فروش و فروخته شده را می‌گیریم
        df_zip = run_scraper(
            location=zip_code,
            listing_types=['for_sale', 'sold'],
            limit=100,
            past_days=365,
            mls_only=True,
            save_csv=False # در حین حلقه فایل جداگانه نسازد
        )
        
        if not df_zip.empty:
            all_scraped_data.append(df_zip)
            # دیتای همین کد پستی رو سریع ذخیره می‌کنیم توی دیتابیس
            upsert_properties(df_zip)
            print(f"✅ {len(df_zip)} ملک از کد پستی {zip_code} با موفقیت ذخیره شد.")
    except Exception as e:
        print(f"❌ خطا در اسکرپ کد پستی {zip_code}: {e}")

# ادغام همه در یک دیتافریم نهایی برای تحلیل
if all_scraped_data:
    df_final = pd.concat(all_scraped_data, ignore_index=True)
    print(f"\n🎉 عملیات به پایان رسید! تعداد کل رکوردهای جدید: {len(df_final)}")
else:
    print("\n❌ متاسفانه هیچ دیتایی استخراج نشد.")

🚀 شروع فرآیند استخراج داده‌های انبوه...

🔍 در حال اسکرپ کد پستی 78701...
✅ 193 ملک از کد پستی 78701 با موفقیت ذخیره شد.

🔍 در حال اسکرپ کد پستی 78702...
✅ 188 ملک از کد پستی 78702 با موفقیت ذخیره شد.

🔍 در حال اسکرپ کد پستی 78703...
✅ 189 ملک از کد پستی 78703 با موفقیت ذخیره شد.

🔍 در حال اسکرپ کد پستی 78704...
✅ 192 ملک از کد پستی 78704 با موفقیت ذخیره شد.

🔍 در حال اسکرپ کد پستی 78705...
✅ 183 ملک از کد پستی 78705 با موفقیت ذخیره شد.

🔍 در حال اسکرپ کد پستی 78741...
✅ 180 ملک از کد پستی 78741 با موفقیت ذخیره شد.

🔍 در حال اسکرپ کد پستی 78744...
✅ 178 ملک از کد پستی 78744 با موفقیت ذخیره شد.

🔍 در حال اسکرپ کد پستی 78745...
✅ 185 ملک از کد پستی 78745 با موفقیت ذخیره شد.

🔍 در حال اسکرپ کد پستی 78750...
✅ 185 ملک از کد پستی 78750 با موفقیت ذخیره شد.

🔍 در حال اسکرپ کد پستی 78759...
✅ 193 ملک از کد پستی 78759 با موفقیت ذخیره شد.

🔍 در حال اسکرپ کد پستی 78717...
✅ 191 ملک از کد پستی 78717 با موفقیت ذخیره شد.

🔍 در حال اسکرپ کد پستی 78723...
✅ 179 ملک از کد پستی 78723 با موفقیت ذخیره شد.


In [3]:
# ذخیره کردن دیتافریم در یک فایل CSV
df_final.to_csv("us_housing_dataset.csv", index=False)
print("✅ فایل با موفقیت ذخیره شد!")

✅ فایل با موفقیت ذخیره شد!
